## Annotation agreement

Uses sklearn.metrics.cohen_kappa_score for Cohen's kappa and weights="linear" for confidence. Binary scores are 0/1; confidence is 1/2/3. Requires pandas, openpyxl, and scikit-learn (listed in the project environment.yml).

Identical constant ratings produce NaN because standard kappa is undefined; scikit-learn may emit a warning for these cases. Weighted kappa is only computed for confidence. Percentage agreement and counts are shown alongside kappa.

Human pairs are grouped without regard to order, and their ratings are aligned by annotator name. The pooled human comparison uses the workbook's two rating positions across all games.

Reads Data/Annotations_w_desc_text_new.xlsx. The five columns prefixed with 1 contain Annotator1's answers, and those prefixed with 2 contain Annotator2's answers. All comparisons use these separate columns.


In [ ]:
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np

workbook = Path("Data/Annotations_w_desc_text.xlsx")
annotations = pd.read_excel(workbook)


In [ ]:
annotations.head()

In [3]:
from sklearn.metrics import cohen_kappa_score
from IPython.display import display

fields = ["IsCoherent", "ClearObjective", "ClearWin", "MultipInteract", "Confidence"]
scales = {field: [0, 1] for field in fields[:-1]}
scales["Confidence"] = [1, 2, 3]

# Prefix 1 belongs to Annotator1; prefix 2 belongs to Annotator2.
first = annotations[[f"1{field}" for field in fields]].copy()
second = annotations[[f"2{field}" for field in fields]].copy()
first.columns = second.columns = fields
for field in fields:
    assert first[field].isin(scales[field]).all(), f"Invalid or missing 1{field} rating"
    assert second[field].isin(scales[field]).all(), f"Invalid or missing 2{field} rating"
first = first.astype(int)
second = second.astype(int)

def compare_scores(left, right):
    """Agreement for two aligned sets of ratings."""
    results = []
    for field in fields:
        a, b = left[field], right[field]
        result = {
            "Field": field,
            "N": len(a),
            "Agreement (%)": 100 * (a.to_numpy() == b.to_numpy()).mean(),
            "Cohen's kappa": cohen_kappa_score(a, b, labels=scales[field]),
        }
        if field == "Confidence":
            result["Linear weighted kappa"] = cohen_kappa_score(
                a, b, labels=scales[field], weights="linear"
            )
        results.append(result)
    return pd.DataFrame(results).set_index("Field")

overall = compare_scores(first, second)
pair_names = annotations[["Annotator1", "Annotator2"]].apply(
    lambda names: " / ".join(sorted(names)), axis=1
)
# Align the scores by name before combining reversed pairs.
reverse = annotations["Annotator1"] > annotations["Annotator2"]
left = first.where(~reverse, second, axis=0)
right = second.where(~reverse, first, axis=0)
pairwise = pd.concat({
    pair: compare_scores(left.loc[indices], right.loc[indices])
    for pair, indices in pair_names.groupby(pair_names).groups.items()
}, names=["Annotator pair"])

def show_comparisons(overall, comparisons):
    display(overall.round(3))
    kappa = comparisons["Cohen's kappa"].unstack("Field").reindex(columns=fields)
    kappa["Confidence (linear weighted)"] = comparisons.xs(
        "Confidence", level="Field"
    )["Linear weighted kappa"]
    for title, table in [
        ("Kappa", kappa.round(3)),
        ("Agreement (%)", comparisons["Agreement (%)"].unstack("Field").reindex(columns=fields).round(1)),
        ("Number of comparisons", comparisons["N"].unstack("Field").reindex(columns=fields)),
    ]:
        print(title)
        display(table)

show_comparisons(overall, pairwise)


,N,Agreement (%),Cohen's kappa,Linear weighted kappa
Field,,,,
IsCoherent,90,98.889,0.851,NaN
ClearObjective,90,86.667,0.627,NaN
ClearWin,90,94.444,0.884,NaN
MultipInteract,90,90.000,0.733,NaN
Confidence,90,83.333,0.740,0.791


Kappa


Field,IsCoherent,ClearObjective,ClearWin,MultipInteract,Confidence,Confidence (linear weighted)
Annotator pair,,,,,,
Cornelius / Elias,NaN,0.824,0.824,0.750,0.875,0.903
Cornelius / Kristoffer,NaN,0.000,1.000,NaN,1.000,1.000
Cornelius / Lilja,NaN,0.250,1.000,0.000,0.429,0.500
Cornelius / Pietro,NaN,1.000,1.000,1.000,0.727,0.786
Elias / Kristoffer,NaN,0.000,0.667,1.000,0.700,0.727
Elias / Lilja,1.0,0.636,0.824,0.625,0.625,0.727
Elias / Pietro,NaN,0.571,0.571,0.667,0.714,0.750
Kristoffer / Lilja,1.0,0.429,1.000,0.657,0.721,0.760
Kristoffer / Pietro,NaN,0.625,0.824,0.800,0.852,0.872


Agreement (%)


Field,IsCoherent,ClearObjective,ClearWin,MultipInteract,Confidence
Annotator pair,,,,,
Cornelius / Elias,100.0,91.7,91.7,91.7,91.7
Cornelius / Kristoffer,100.0,83.3,100.0,100.0,100.0
Cornelius / Lilja,100.0,66.7,100.0,83.3,66.7
Cornelius / Pietro,100.0,100.0,100.0,100.0,83.3
Elias / Kristoffer,100.0,83.3,83.3,100.0,83.3
Elias / Lilja,100.0,83.3,91.7,83.3,75.0
Elias / Pietro,100.0,83.3,83.3,83.3,83.3
Kristoffer / Lilja,100.0,83.3,100.0,83.3,83.3
Kristoffer / Pietro,100.0,91.7,91.7,91.7,91.7


Number of comparisons


Field,IsCoherent,ClearObjective,ClearWin,MultipInteract,Confidence
Annotator pair,,,,,
Cornelius / Elias,12,12,12,12,12
Cornelius / Kristoffer,6,6,6,6,6
Cornelius / Lilja,6,6,6,6,6
Cornelius / Pietro,12,12,12,12,12
Elias / Kristoffer,6,6,6,6,6
Elias / Lilja,12,12,12,12,12
Elias / Pietro,6,6,6,6,6
Kristoffer / Lilja,12,12,12,12,12
Kristoffer / Pietro,12,12,12,12,12


## Human agreement with AI

Games are matched by BGGId against ai_annotation/games_annotated.xlsx. Each human score is compared with the AI score for that game. The pooled result includes two human-AI comparisons per game (180 comparisons across 90 games); these comparisons share the same AI rating within each game. Each annotator is also compared separately, using every game they rated.

Both percentage agreement and Cohen's kappa are shown. Confidence additionally uses linear weighted kappa on the 1 to 3 scale. Standard Cohen's kappa is used; identical constant ratings give an undefined kappa. Weighted kappa is only calculated for confidence, so other fields show Not applicable in that column.

The AI labeling script includes detailed criteria and automatic checks for ClearWin and Confidence. These comparisons measure agreement with the saved AI labels; they do not establish which annotator is correct.


In [5]:
ai_annotations = pd.read_excel("ai_annotation/games_annotated.xlsx", sheet_name="annotations")
assert annotations["BGGId"].is_unique and ai_annotations["BGGId"].is_unique
assert set(annotations["BGGId"]) == set(ai_annotations["BGGId"])
ai = ai_annotations.set_index("BGGId").loc[annotations["BGGId"], fields].reset_index(drop=True)
for field in fields:
    assert ai[field].isin(scales[field]).all(), f"Invalid AI score in {field}"

human_scores = pd.concat([first, second], ignore_index=True)
ai_scores = pd.concat([ai, ai], ignore_index=True)
annotators = pd.concat(
    [annotations["Annotator1"], annotations["Annotator2"]], ignore_index=True
)
assert annotators.notna().all()
human_ai_overall = compare_scores(human_scores, ai_scores)
human_ai_by_annotator = pd.concat({
    name: compare_scores(human_scores.loc[indices], ai_scores.loc[indices])
    for name, indices in annotators.groupby(annotators).groups.items()
}, names=["Human annotator"])

show_comparisons(human_ai_overall, human_ai_by_annotator)


,N,Agreement (%),Cohen's kappa,Linear weighted kappa
Field,,,,
IsCoherent,180,96.111,0.000,NaN
ClearObjective,180,85.556,0.526,NaN
ClearWin,180,88.333,0.758,NaN
MultipInteract,180,82.778,0.492,NaN
Confidence,180,73.889,0.580,0.653


Kappa


Field,IsCoherent,ClearObjective,ClearWin,MultipInteract,Confidence,Confidence (linear weighted)
Human annotator,,,,,,
Cornelius,NaN,0.442,0.809,0.600,0.497,0.609
Elias,0.0,0.591,0.613,0.556,0.562,0.631
Kristoffer,0.0,0.640,0.818,0.491,0.611,0.653
Lilja,0.0,0.325,0.827,0.264,0.470,0.557
Pietro,NaN,0.722,0.719,0.557,0.728,0.781


Agreement (%)


Field,IsCoherent,ClearObjective,ClearWin,MultipInteract,Confidence
Human annotator,,,,,
Cornelius,100.0,80.6,91.7,88.9,69.4
Elias,94.4,86.1,80.6,83.3,72.2
Kristoffer,97.2,94.4,91.7,83.3,77.8
Lilja,88.9,75.0,91.7,75.0,66.7
Pietro,100.0,91.7,86.1,83.3,83.3


Number of comparisons


Field,IsCoherent,ClearObjective,ClearWin,MultipInteract,Confidence
Human annotator,,,,,
Cornelius,36,36,36,36,36
Elias,36,36,36,36,36
Kristoffer,36,36,36,36,36
Lilja,36,36,36,36,36
Pietro,36,36,36,36,36
